In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

os.chdir("/content/drive/MyDrive/SUB^2/Character-level-Tagging")

In [11]:
!ls srcs

conll_utils.py	korean_morph_seq2.py	well_formed_filter.py
ext_korean.py	projectivize_filter.py
__init__.py	__pycache__


# Module

In [4]:
!pip install jamo

In [5]:
# 📦 라이브러리 불러오기

import sys
import json
import jamo
from tqdm import tqdm
import unicodedata
import shutil
import random
import re
import argparse
import logging

In [23]:
# 📦 모듈 불러오기

from srcs.conll_utils import *
from srcs.korean_morph_seq2 import *

# Preprocessing

In [24]:
def convert_ud(input_file, output_file):
    with open(input_file, 'rb') as f_in:
        data = json.load(f_in)

    output_dir = os.path.dirname(output_file)
    os.makedirs(output_dir, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f_out:
        for doc in data['document']:
            for sent in doc['sentence']:
                if len(sent['form']) == 0: continue
                outID = 1
                words = sent['word']
                morphs = sent.get('morpheme', [])
                # word_id → list of morphs
                morph_dict = {}
                if morphs is None:
                    print(doc['sentence'])
                for m in morphs:
                    wid = m['word_id']
                    if wid not in morph_dict:
                        morph_dict[wid] = []
                    morph = m['form']
                    pos = m['label']
                    morph_dict[wid].append(f"{morph}/POS={pos}".lower())

                for word in words:
                    wid = word['id']
                    form = word['form']

                    morph_string = ' + '.join(morph_dict.get(wid, []))
                    # 출력 포맷: ID, 어절, 형태소+태그, 그 외 7개 _
                    out_line = f"{outID}\t{form}\t{morph_string}\t_\t_\t_\t_\t_\t_\n"
                    f_out.write(out_line)
                    outID += 1

                # 문장 구분 공백 줄
                f_out.write('\n')

In [25]:
input_dir = "dataset/raw_data" ## Directory of Raw File

i = 0
for path in os.listdir(input_dir):
    if not path.endswith("json"): continue
    print(path)
    i += 1

    convert_ud(
        input_file = os.path.join(input_dir, path),
        output_file = f"dataset/preprocessed/NILK_MP_{i}"  ## Directory of Preprocessed File
    )

NXMP1902008040.json
SXMP1902008031.json


# Get Character Tag

## Parameter

In [26]:
## Directory ##

corpus_files = ['dataset/preprocessed/NILK_MP_1', 'dataset/preprocessed/NILK_MP_2']  ## Directory of preprocessed File
output_dir = 'dataset/procfile'   ## Directory of processed File
os.makedirs(output_dir, exist_ok=True)

## 전처리 제약 조건 ##
min_word_count = 0
max_char_count = 400

In [27]:
class OutputUnit(object):
    def __init__(self):
        # char itself
        self.c = ''
        # can be used as secondary input to DNN to specify word-breaks, etc
        self.flag = ''
        self.action_tag = ''

class OutputMorph(object):
    def __init__(self):
        # used during output
        # morpheme itself
        self.form = ''
        # can be used as secondary input to DNN to specify word-breaks, etc
        self.flag = ''

        # original untransformed word that this morpheme is a part of
        # can be used as auxiliary input to POS tagger to make use of word
        # vectors
        # self.orig_word = ''
        # this class is stored inside a word, so we can use that instead

        self.action_tag = ''

class OutputWord(object):
    def __init__(self):
        # used during output
        # word itself
        self.form = ''
        # can be used as secondary input to DNN to specify word-breaks, etc
        self.flag = ''
        self.action_tag = ''

        # for marmot
        self.pos_tag = ''

        # if in unit2unit2 mode only
        self.output_units = []

        # list of output morphemes in the word
        # mode 2 only
        self.output_morphs = []

    def unitAppend(self, c):
        self.output_units.append(c)

    def morphAppend(self, m):
        self.output_morphs.append(m)

class OutputSentence(object):
    def __init__(self):
        # list of output words in the sentence
        self.output_words = []

        # pointer to orig conll sentence (used during filtering)
        self.orig_conll_sentence = None

    def append(self, w):
        self.output_words.append(w)

In [28]:
## 전처리 ##
space_tbl = [chr(i) for i in range(sys.maxunicode)
                      if unicodedata.category(chr(i)).startswith('Zs')]

def strip_all_space(s):
    result = ''
    for c in s.strip():
        if c in space_tbl:
            pass
        else:
            result += c
    return result.strip()

## 태그 설정 ##
def actionToPrettyString(a):
    if type(a) is tuple:
        if a[0] == 'KEEP':
            return 'KEEP'
        elif a[0] == 'NOOP':
            return 'NOOP'
        elif a[0] == 'B-KEEP':
            return 'B-KEEP'
        elif a[0] == 'I-KEEP':
            return 'I-KEEP'
        elif a[0] == 'MOD':
            #return 'MOD:' + ''.join(a[1])
            return 'MOD:' + str(a[1]).replace(', ', ',')
        else:
            assert None, 'unknown action: ' + str(a)
    else:
        assert ' ' not in a
        return a # just trust it

## Processing

In [29]:
def procFile(fname):
    global dupes
    global sent_error_count
    ### 파일 로드 ###
    fd = open(fname, 'r', encoding='utf-8')
    contents = fd.read()
    fd.close()

    retval_sentences = []

    corpus = ConllFile(keepMalformed=True,
                    checkParserConformity=False,
                    projectivize=False,
                    enableLemmaMorphemes=True,
                    enableLemmaAsStem=False,
                    compatibleJamo=True,
                    fixMorphemeLabelBugs=True,
                    spmrlFormat=False,
                    koreanAsJamoSeq=False)
    corpus.read(contents)

    for sentence_idx, sentence in enumerate(corpus):
        sentence.evict = False
        error_found = False

        total_chars = 0

        ### 중복 제거 ###
        h = hash(sentence)
        if h in dupes:
            sentence_idx = dupes[h]
            if corpus.sentences[sentence_idx].is_equal(sentence):
                sentence.evict = True
                sent_error_count += 1
                continue
            else:
                pass
        else:
            dupes[h] = sentence_idx

        ### 오류 탐색 ###
        for wd_i, wd in enumerate(sentence.tokens):
            if wd.FORM:
                if len(wd.FORM.strip()) == 0 or ' ' in wd.FORM:
                    error_found = True

                form_copy = wd.FORM

                for c in form_copy:
                    total_chars += 1
                    if len(strip_all_space(c)) == 0:
                        error_found = True
            else:
                error_found = True

            if not error_found:
                if (not wd.LEMMA) or len(wd.LEMMA.strip()) == 0:
                    error_found = True

            if not error_found:
                if wd.morphemes:
                    for m in wd.morphemes:
                        if len(strip_all_space(m[0])) == 0:
                            error_found = True
                        if len(strip_all_space(m[1])) == 0:
                            error_found = True

        if error_found:
            sentence.evict = True
            sent_error_count += 1
            continue

        if max_char_count != 0:
            if total_chars > max_char_count:
                sentence.evict = True
                sent_error_count += 1
                continue


        current_sentence = OutputSentence()
        current_sentence.orig_conll_sentence = sentence

        for wd_i in range(len(sentence.tokens)):
            wd = sentence.tokens[wd_i]
            current_word = OutputWord()
            current_word.form = wd.FORM

            if wd.UPOSTAG == None:
                current_word.pos_tag = ''
            else:
                current_word.pos_tag = wd.UPOSTAG.lower()

            segs = [m[0] for m in wd.morphemes]

            try:
                a = greedyOracle(current_word.form, segs)
            except:
                print('failed', wd_i, current_word.form, segs,
                        sentence.toFileOutput())
                sys.exit(1)

            action_list = addSegmentation(a, current_word.form, segs)
            c = restoreOrigSegments(action_list, current_word.form)
            assert c == segs, 'restored segments don\'t match: ' + str(c) + ' vs ' + str(segs)
            assert len(action_list) == len(current_word.form)

            for i in range(len(current_word.form)):
                current_unit = OutputUnit()
                a = action_list[i]
                f = current_word.form[i]
                current_unit.c = f

                assert len(strip_all_space(f)) != 0

                action = actionToPrettyString(a)

                current_unit.action_tag = action

                current_word.unitAppend(current_unit)

            if min_word_count != 0:
                if len(current_sentence) >= min_word_count:
                    current_sentence.append(current_word)
                else:
                    sentence.evict = True
            else:
                current_sentence.append(current_word)

        retval_sentences.append(current_sentence)

    return retval_sentences

In [30]:
dupes = dict()
sent_error_count = 0
all_sentences = []

for f in corpus_files:
    print('Processing %s...' % f)
    all_sentences += procFile(f)

Processing dataset/preprocessed/NILK_MP_1...
Processing dataset/preprocessed/NILK_MP_2...


In [31]:
print(len(all_sentences))
print(sent_error_count)

330005
41566


In [32]:
def getFileOutput(sentences):
    fileOutput = ''

    outSentences = []
    for sent_idx, sent in enumerate(all_sentences):
        sent.internal_idx = sent_idx
        if sent.orig_conll_sentence.evict:
            continue
        curSentence = []
        for word in sent.output_words:
            for unit in word.output_units:
                curSentence.append('%s\t%s\t%s\t%s' % (unit.c,
                                                        '',
                                                        '',
                                                        unit.action_tag))

        outSentences.append(curSentence)
    fileOutput = '\n\n'.join(['\n'.join(sent) for sent in outSentences])
    return fileOutput

In [33]:
output_fn = os.path.join(output_dir, 'char2morph.txt')
f = open(output_fn, 'w', encoding='utf-8')
f.write(getFileOutput(all_sentences))
f.close()

# Statistics

In [34]:
!pip install hgtk

In [35]:
from tqdm import tqdm
import numpy as np
import hgtk

In [36]:
with open("dataset/procfile/char2morph.txt", 'r') as f:
    result = f.readlines()

In [37]:
c_num = 0
total_count = {"KEEP": 0, "NOOP": 0, "MOD": 0} ## 한국어만

char_cnt = 0
subchar_cnt = 0


with tqdm(total = len(result)) as pbar:
    for i, line in enumerate(result):
        if len(line.strip()) == 0:
            continue
        temp = line.strip().split('\t\t\t')
        if len(temp) != 2:
            raise ValueError("Morphs split Error!")
        char, morph = temp[0], temp[1]
        if len(char) != 1:
            raise ValueError("Char is not Character!")

        c_num += 1

        if hgtk.checker.is_hangul(char):
            if morph.endswith("KEEP"):
                total_count["KEEP"] += 1
            elif morph.endswith("NOOP"):
                total_count["NOOP"] += 1
            else:
                total_count["MOD"] += 1
                if char in morph:
                    char_cnt += 1
                else:
                    subchar_cnt += 1

        pbar.update(1)

 97%|█████████▋| 9430663/9760667 [00:21<00:00, 432698.03it/s]


In [38]:
print(f"전체 character 개수 : {c_num} ")
print(f"한국어 character 개수 : {sum(total_count.values())} ({np.round(sum(total_count.values()) / c_num * 100, 2)} [%])")
print(f"\t- KEEP 개수 : {total_count['KEEP']} ({np.round(total_count['KEEP'] / c_num * 100, 2)} [%])")
print(f"\t- NOOP 개수 : {total_count['NOOP']} ({np.round(total_count['NOOP'] / c_num * 100, 2)} [%])")
print(f"\t- MOD 개수 : {total_count['MOD']} ({np.round(total_count['MOD'] / c_num * 100, 2)} [%])")
print(f"\t\t- CHAR MOD 개수 : {char_cnt} ({np.round(char_cnt / total_count['MOD'] * 100, 2)} [%])")
print(f"\t\t- SUBCHAR MOD 개수 : {subchar_cnt} ({np.round(subchar_cnt / total_count['MOD'] * 100, 2)} [%])")

전체 character 개수 : 9430663 
한국어 character 개수 : 8524016 (90.39 [%])
	- KEEP 개수 : 8011726 (84.95 [%])
	- NOOP 개수 : 29241 (0.31 [%])
	- MOD 개수 : 483049 (5.12 [%])
		- CHAR MOD 개수 : 35019 (7.25 [%])
		- SUBCHAR MOD 개수 : 448030 (92.75 [%])
